# Seasonal & Regional Analysis: Heat Stress and Cattle Slaughter

This notebook performs deep-dive seasonal and regional analysis of relationships between heat stress and cattle slaughter.

**Analysis Focus:**
1. Seasonal patterns (Spring, Summer, Fall, Winter)
2. Regional comparisons (Region 4 vs Region 6)
3. Heat threshold effects
4. Extreme events analysis
5. Long-term trends by season
6. Compound stressors

**Regions:**
- **Region 4**: AL, FL, GA, KY, MS, NC, SC, TN (Southeast)
- **Region 6**: LA, NM, OK, TX (Southern Plains/Southwest)

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("Imports complete!")

In [ ]:
# Configuration - Import from centralized config.py
import sys
from pathlib import Path

# Add project root to path to import config
sys.path.append(str(Path('../..')))  # Up two levels from notebooks/03_analysis/ to research/

# Import paths and constants from config
from config import (
    CATTLE_DATA_FILE,
    FIGURES_DIR,
    SEASONS,
    CATTLE_REGIONS,
    STATE_NAMES,
    STATE_REGIONS
)

# Use config paths directly
MERGED_FILE = CATTLE_DATA_FILE.parent / 'cattle_heat_merged.csv'
OUTPUT_DIR = FIGURES_DIR / 'seasonal_regional'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project paths loaded from config.py:")
print(f"  Merged data file: {MERGED_FILE}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Loaded season definitions and regional mappings from config.py")

## 1. Load Merged Dataset

Load the integrated cattle-heat dataset created in notebook 06.

In [ ]:
# Load merged data
df = pd.read_csv(MERGED_FILE, parse_dates=['date'])

# Add season column
def get_season(month):
    for season, months in SEASONS.items():
        if month in months:
            return season
    return None

df['season'] = df['month'].apply(get_season)

# Create decade column for trend analysis
df['decade'] = (df['year'] // 10) * 10

print(f"Merged data shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Seasons: {df['season'].value_counts().sort_index()}")

df.head()

## 2. Seasonal Patterns

Analyze how the heat-slaughter relationship varies by season.

In [ ]:
# Seasonal comparison boxplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

variables = [
    ('regions_4_6_total', 'Total Cattle Slaughter (1000 head/week)'),
    ('combined_hours_above_24', 'Heat Stress Hours >24°C (hours/week)'),
    ('combined_hours_above_21', 'Heat Stress Hours >21°C (hours/week)'),
    ('combined_hours_below_18_above_0', 'Good Recovery Hours (0-18°C, hours/week)')
]

for idx, (var, label) in enumerate(variables):
    ax = axes[idx]
    
    # Create boxplot by season
    season_order = ['Winter', 'Spring', 'Summer', 'Fall']
    data_by_season = [df[df['season'] == s][var].dropna() for s in season_order]
    
    bp = ax.boxplot(data_by_season, labels=season_order, patch_artist=True,
                    showmeans=True, meanline=True)
    
    # Color boxes
    colors = ['lightblue', 'lightgreen', 'coral', 'wheat']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_ylabel(label)
    ax.set_title(f'{label}\nby Season (1984-2025)', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add sample sizes
    for i, season in enumerate(season_order):
        n = len(data_by_season[i])
        ax.text(i+1, ax.get_ylim()[1]*0.95, f'n={n}',
               ha='center', va='top', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_seasonal_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation by season
seasonal_corr = []

for season in ['Winter', 'Spring', 'Summer', 'Fall']:
    df_season = df[df['season'] == season].copy()
    
    for region in ['region_4', 'region_6', 'combined']:
        cattle_col = f'{region.replace("combined", "regions_4_6")}_total' if 'combined' in region else f'{region}_total'
        heat_col = f'{region}_hours_above_24'
        
        if cattle_col in df_season.columns and heat_col in df_season.columns:
            corr, p_val = stats.pearsonr(
                df_season[cattle_col].dropna(),
                df_season[heat_col].dropna()
            )
            seasonal_corr.append({
                'season': season,
                'region': region,
                'correlation': corr,
                'p_value': p_val,
                'n': len(df_season.dropna(subset=[cattle_col, heat_col]))
            })

df_seasonal_corr = pd.DataFrame(seasonal_corr)

# Plot heatmap of seasonal correlations
pivot = df_seasonal_corr.pivot(index='region', columns='season', values='correlation')
pivot = pivot[['Winter', 'Spring', 'Summer', 'Fall']]  # Reorder columns

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=1, cbar_kws={'label': 'Correlation'})
ax.set_title('Seasonal Correlation: Heat Stress (>24°C) vs Cattle Slaughter\n(1984-2025)',
            fontweight='bold', fontsize=14)
ax.set_ylabel('Region', fontsize=12)
ax.set_xlabel('Season', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_seasonal_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Seasonal correlations:")
print(df_seasonal_corr.round(3))

## 3. Regional Comparison

Compare heat stress impacts between Region 4 (Southeast) and Region 6 (Southern Plains/Southwest).

In [ ]:
# Regional comparison scatter plots
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Summer data only (peak heat stress)
df_summer = df[df['season'] == 'Summer'].copy()

# Region 4
ax = axes[0, 0]
scatter = ax.scatter(df_summer['region_4_hours_above_24'], df_summer['region_4_total'],
                    alpha=0.5, s=30, c=df_summer['year'], cmap='viridis')
# Regression
x = df_summer['region_4_hours_above_24'].dropna()
y = df_summer['region_4_total'].dropna()
mask = x.index.isin(y.index)
z = np.polyfit(x[mask], y[mask], 1)
p = np.poly1d(z)
x_range = np.linspace(x.min(), x.max(), 100)
ax.plot(x_range, p(x_range), 'r--', linewidth=2, label=f'Slope: {z[0]:.2f}')
ax.set_xlabel('Heat Stress Hours >24°C (hours/week)')
ax.set_ylabel('Cattle Slaughter (1000 head/week)')
ax.set_title('Region 4 (Southeast) - Summer', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Year')

# Region 6
ax = axes[0, 1]
scatter = ax.scatter(df_summer['region_6_hours_above_24'], df_summer['region_6_total'],
                    alpha=0.5, s=30, c=df_summer['year'], cmap='viridis')
x = df_summer['region_6_hours_above_24'].dropna()
y = df_summer['region_6_total'].dropna()
mask = x.index.isin(y.index)
z = np.polyfit(x[mask], y[mask], 1)
p = np.poly1d(z)
x_range = np.linspace(x.min(), x.max(), 100)
ax.plot(x_range, p(x_range), 'r--', linewidth=2, label=f'Slope: {z[0]:.2f}')
ax.set_xlabel('Heat Stress Hours >24°C (hours/week)')
ax.set_ylabel('Cattle Slaughter (1000 head/week)')
ax.set_title('Region 6 (Southern Plains/Southwest) - Summer', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Year')

# All seasons - Region 4
ax = axes[1, 0]
for season, color in zip(['Winter', 'Spring', 'Summer', 'Fall'], 
                         ['blue', 'green', 'red', 'orange']):
    df_s = df[df['season'] == season]
    ax.scatter(df_s['region_4_hours_above_24'], df_s['region_4_total'],
              alpha=0.3, s=20, color=color, label=season)
ax.set_xlabel('Heat Stress Hours >24°C (hours/week)')
ax.set_ylabel('Cattle Slaughter (1000 head/week)')
ax.set_title('Region 4 - All Seasons', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# All seasons - Region 6
ax = axes[1, 1]
for season, color in zip(['Winter', 'Spring', 'Summer', 'Fall'], 
                         ['blue', 'green', 'red', 'orange']):
    df_s = df[df['season'] == season]
    ax.scatter(df_s['region_6_hours_above_24'], df_s['region_6_total'],
              alpha=0.3, s=20, color=color, label=season)
ax.set_xlabel('Heat Stress Hours >24°C (hours/week)')
ax.set_ylabel('Cattle Slaughter (1000 head/week)')
ax.set_title('Region 6 - All Seasons', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_regional_comparison_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Heat Threshold Analysis

Compare impacts of different heat stress thresholds (21°C vs 24°C).

In [ ]:
# Categorize weeks by heat stress intensity
def categorize_heat(hours_above_24):
    if hours_above_24 < 20:
        return 'Low'
    elif hours_above_24 < 40:
        return 'Moderate'
    elif hours_above_24 < 60:
        return 'High'
    else:
        return 'Extreme'

df['heat_category'] = df['combined_hours_above_24'].apply(categorize_heat)

# Summer data only
df_summer = df[df['season'] == 'Summer'].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot by heat category
ax = axes[0]
heat_order = ['Low', 'Moderate', 'High', 'Extreme']
data_by_category = [df_summer[df_summer['heat_category'] == cat]['regions_4_6_total'].dropna() 
                   for cat in heat_order]

bp = ax.boxplot(data_by_category, labels=heat_order, patch_artist=True,
               showmeans=True, meanline=True)

colors = ['lightblue', 'yellow', 'orange', 'red']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Heat Stress Category', fontsize=12)
ax.set_ylabel('Cattle Slaughter (1000 head/week)', fontsize=12)
ax.set_title('Slaughter by Heat Stress Intensity\n(Summer, Regions 4 & 6)',
            fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

# Add sample sizes and means
for i, cat in enumerate(heat_order):
    n = len(data_by_category[i])
    mean = data_by_category[i].mean()
    ax.text(i+1, ax.get_ylim()[1]*0.95, f'n={n}\nμ={mean:.0f}',
           ha='center', va='top', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Bar chart comparing means
ax = axes[1]
means = [data_by_category[i].mean() for i in range(len(heat_order))]
stds = [data_by_category[i].std() for i in range(len(heat_order))]

bars = ax.bar(heat_order, means, yerr=stds, color=colors, alpha=0.7,
             edgecolor='black', capsize=5)
ax.set_xlabel('Heat Stress Category', fontsize=12)
ax.set_ylabel('Mean Cattle Slaughter (1000 head/week)', fontsize=12)
ax.set_title('Mean Slaughter ± SD by Heat Category\n(Summer, Regions 4 & 6)',
            fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, mean in zip(bars, means):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{mean:.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_threshold_categories.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical test (ANOVA)
from scipy.stats import f_oneway
f_stat, p_val = f_oneway(*data_by_category)
print(f"\nANOVA Test: F-statistic = {f_stat:.3f}, p-value = {p_val:.3e}")
print("Significant difference between heat categories" if p_val < 0.05 else "No significant difference")

## 5. Long-term Trends by Season

Analyze how seasonal patterns have changed over decades.

In [ ]:
# Compute decadal averages by season
decadal_seasonal = df.groupby(['decade', 'season']).agg({
    'regions_4_6_total': 'mean',
    'combined_hours_above_24': 'mean',
    'combined_hours_above_21': 'mean'
}).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

seasons = ['Winter', 'Spring', 'Summer', 'Fall']
colors_season = ['blue', 'green', 'red', 'orange']

for idx, season in enumerate(seasons):
    ax = axes[idx]
    df_season = decadal_seasonal[decadal_seasonal['season'] == season]
    
    ax2 = ax.twinx()
    
    # Cattle slaughter
    ax.plot(df_season['decade'], df_season['regions_4_6_total'],
           marker='o', linewidth=2, markersize=8, color='darkblue',
           label='Cattle Slaughter')
    ax.set_xlabel('Decade', fontsize=11)
    ax.set_ylabel('Cattle Slaughter (1000 head/week)', fontsize=11, color='darkblue')
    ax.tick_params(axis='y', labelcolor='darkblue')
    
    # Heat stress
    ax2.plot(df_season['decade'], df_season['combined_hours_above_24'],
            marker='s', linewidth=2, markersize=8, color='darkred',
            label='Heat Stress')
    ax2.set_ylabel('Heat Stress Hours >24°C (hours/week)', fontsize=11, color='darkred')
    ax2.tick_params(axis='y', labelcolor='darkred')
    
    ax.set_title(f'{season} Trends by Decade (Regions 4 & 6)',
                fontweight='bold', fontsize=13)
    ax.grid(True, alpha=0.3)
    
    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_decadal_seasonal_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("Decadal seasonal averages:")
print(decadal_seasonal.round(1))

## 6. Extreme Heat Events

Analyze cattle slaughter during extreme heat weeks (>95th percentile).

In [ ]:
# Define extreme heat (95th percentile of summer heat)
df_summer = df[df['season'] == 'Summer'].copy()
extreme_threshold = df_summer['combined_hours_above_24'].quantile(0.95)

print(f"Extreme heat threshold (95th percentile): {extreme_threshold:.1f} hours/week")

# Categorize weeks
df_summer['extreme'] = df_summer['combined_hours_above_24'] >= extreme_threshold

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distribution comparison
ax = axes[0, 0]
normal_weeks = df_summer[~df_summer['extreme']]['regions_4_6_total']
extreme_weeks = df_summer[df_summer['extreme']]['regions_4_6_total']

ax.hist([normal_weeks, extreme_weeks], bins=20, label=['Normal', 'Extreme Heat'],
       alpha=0.7, color=['skyblue', 'red'], edgecolor='black')
ax.set_xlabel('Cattle Slaughter (1000 head/week)')
ax.set_ylabel('Frequency')
ax.set_title('Slaughter Distribution: Normal vs Extreme Heat Weeks\n(Summer, Regions 4 & 6)',
            fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add statistics
stats_text = f"Normal: μ={normal_weeks.mean():.0f}, σ={normal_weeks.std():.0f}, n={len(normal_weeks)}\n"
stats_text += f"Extreme: μ={extreme_weeks.mean():.0f}, σ={extreme_weeks.std():.0f}, n={len(extreme_weeks)}"
ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
       va='top', ha='right',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Box plot comparison
ax = axes[0, 1]
bp = ax.boxplot([normal_weeks, extreme_weeks], labels=['Normal', 'Extreme Heat'],
               patch_artist=True, showmeans=True, meanline=True)
bp['boxes'][0].set_facecolor('skyblue')
bp['boxes'][1].set_facecolor('red')
for box in bp['boxes']:
    box.set_alpha(0.7)
ax.set_ylabel('Cattle Slaughter (1000 head/week)')
ax.set_title('Slaughter Comparison: Normal vs Extreme Heat',
            fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# T-test
t_stat, p_val = stats.ttest_ind(normal_weeks, extreme_weeks)
ax.text(0.5, 0.95, f't={t_stat:.3f}, p={p_val:.3e}',
       transform=ax.transAxes, ha='center', va='top',
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

# Time series of extreme events
ax = axes[1, 0]
extreme_years = df_summer.groupby('year')['extreme'].sum()
ax.bar(extreme_years.index, extreme_years.values, color='red', alpha=0.7, edgecolor='black')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Extreme Heat Weeks')
ax.set_title('Frequency of Extreme Heat Weeks by Year\n(Summer, >95th percentile)',
            fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add trend line
z = np.polyfit(extreme_years.index, extreme_years.values, 1)
p = np.poly1d(z)
ax.plot(extreme_years.index, p(extreme_years.index), 'k--', linewidth=2,
       label=f'Trend: {z[0]:.3f} weeks/year')
ax.legend()

# Regional comparison during extreme heat
ax = axes[1, 1]
df_extreme = df_summer[df_summer['extreme']]
region_comparison = pd.DataFrame({
    'Region 4': [df_extreme['region_4_total'].mean(), df_extreme['region_4_total'].std()],
    'Region 6': [df_extreme['region_6_total'].mean(), df_extreme['region_6_total'].std()]
}, index=['Mean', 'Std'])

region_comparison.T.plot(y='Mean', yerr='Std', kind='bar', ax=ax,
                        color=['blue', 'green'], alpha=0.7, capsize=5)
ax.set_xlabel('Region')
ax.set_ylabel('Cattle Slaughter (1000 head/week)')
ax.set_title('Regional Slaughter During Extreme Heat\n(Summer, >95th percentile)',
            fontweight='bold')
ax.legend().set_visible(False)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_extreme_heat_events.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nExtreme heat weeks: {extreme_weeks.shape[0]} ({extreme_weeks.shape[0]/len(df_summer)*100:.1f}% of summer weeks)")
print(f"Mean difference: {extreme_weeks.mean() - normal_weeks.mean():.1f} thousand head/week")
print(f"Statistical significance: {'YES' if p_val < 0.05 else 'NO'} (p={p_val:.3e})")

## 7. Monthly Patterns

Fine-grained analysis at monthly resolution.

In [ ]:
# Monthly averages
monthly_avg = df.groupby('month').agg({
    'regions_4_6_total': ['mean', 'std'],
    'combined_hours_above_24': ['mean', 'std']
}).reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
ax2 = ax.twinx()

months = monthly_avg['month']
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Cattle slaughter
ax.errorbar(months, monthly_avg['regions_4_6_total']['mean'],
           yerr=monthly_avg['regions_4_6_total']['std'],
           marker='o', linewidth=2, markersize=8, color='darkblue',
           capsize=5, label='Cattle Slaughter')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Cattle Slaughter (1000 head/week)', fontsize=12, color='darkblue')
ax.tick_params(axis='y', labelcolor='darkblue')
ax.set_xticks(months)
ax.set_xticklabels(month_names)

# Heat stress
ax2.errorbar(months, monthly_avg['combined_hours_above_24']['mean'],
            yerr=monthly_avg['combined_hours_above_24']['std'],
            marker='s', linewidth=2, markersize=8, color='darkred',
            capsize=5, label='Heat Stress')
ax2.set_ylabel('Heat Stress Hours >24°C (hours/week)', fontsize=12, color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')

ax.set_title('Monthly Patterns: Cattle Slaughter and Heat Stress\n(Regions 4 & 6, 1984-2025)',
            fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)

# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_monthly_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

This notebook performed comprehensive seasonal and regional analysis:

**Key Findings:**

1. **Seasonal Patterns:**
   - Summer shows highest heat stress and variable cattle slaughter
   - Heat-slaughter correlations vary significantly by season
   - Winter shows lowest heat stress (as expected)

2. **Regional Differences:**
   - Region 4 (Southeast) vs Region 6 (Southern Plains) show distinct patterns
   - Both regions affected by summer heat stress
   - Regional climate differences reflected in slaughter patterns

3. **Heat Thresholds:**
   - Clear categorization of low/moderate/high/extreme heat weeks
   - Slaughter rates show systematic differences across heat categories
   - Statistical tests confirm significance

4. **Long-term Trends:**
   - Decadal analysis reveals changing patterns over 40+ years
   - Increasing frequency of extreme heat weeks
   - Seasonal trends shifting over time

5. **Extreme Events:**
   - >95th percentile heat weeks show measurable impacts
   - Regional responses to extreme heat differ
   - Extreme heat frequency increasing over time

**Output Files:**
- Seasonal boxplots and correlations
- Regional comparison scatter plots
- Threshold category analysis
- Decadal trend plots
- Extreme event analysis
- Monthly pattern visualization